# Part C — GAN (Generative Adversarial Network)
### Generate Fashion-MNIST Images
**Models:** Generator + Discriminator  
**Dataset:** Fashion-MNIST  
**Run Time:** ~5-8 minutes total on CPU

In [ ]:
# CELL 1 — Imports (no install needed — already installed)
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import os, time

torch.manual_seed(42)
print('✅ All imports done!')

In [ ]:
# CELL 2 — Settings
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS      = 30         # ✅ Only 30 epochs — fast on CPU
BATCH_SIZE  = 128        # ✅ Large batch = fewer steps per epoch
LR          = 0.0002     # standard GAN learning rate
NOISE_DIM   = 64         # ✅ Small noise vector = faster Generator
SAVE_EVERY  = 5          # save generated images every 5 epochs

os.makedirs('./outputs',          exist_ok=True)
os.makedirs('./generated_images', exist_ok=True)

print(f'✅ Device     : {DEVICE}')
print(f'✅ Epochs     : {EPOCHS}')
print(f'✅ Batch Size : {BATCH_SIZE}')
print(f'✅ Noise Dim  : {NOISE_DIM}')

In [ ]:
# CELL 3 — Load Fashion-MNIST Dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])   # normalize to [-1, 1] for tanh
])

print('📥 Downloading Fashion-MNIST...')
dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)

# ✅ Use only 10,000 samples for faster training
subset  = torch.utils.data.Subset(dataset, range(10000))
loader  = torch.utils.data.DataLoader(
    subset, batch_size=BATCH_SIZE, shuffle=True
)

print(f'✅ Dataset ready — using {len(subset)} samples')

# Show sample real images
imgs, _ = next(iter(loader))
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('Real Fashion-MNIST Images', fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.savefig('./outputs/real_samples.png', dpi=150)
plt.show()
print('✅ Saved: real_samples.png')

In [ ]:
# CELL 4 — Build Generator
# Takes random noise → produces fake 28x28 image
class Generator(nn.Module):
    def __init__(self, noise_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            # noise_dim → 128
            nn.Linear(noise_dim, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),

            # 128 → 256
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),

            # 256 → 512
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),

            # 512 → 784 (28x28 image)
            nn.Linear(512, 784),
            nn.Tanh()            # output in [-1, 1]
        )

    def forward(self, z):
        img = self.net(z)                    # (batch, 784)
        return img.view(-1, 1, 28, 28)       # reshape to image


G = Generator(noise_dim=NOISE_DIM).to(DEVICE)
print('✅ Generator ready!')
print(f'   Parameters: {sum(p.numel() for p in G.parameters()):,}')
print(G)

In [ ]:
# CELL 5 — Build Discriminator
# Takes 28x28 image → outputs real(1) or fake(0)
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # 784 → 256
            nn.Linear(784, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 256 → 128
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 128 → 1 (real or fake probability)
            nn.Linear(128, 1),
            nn.Sigmoid()         # output between 0 and 1
        )

    def forward(self, img):
        img = img.view(-1, 784)  # flatten image
        return self.net(img)     # (batch, 1)


D = Discriminator().to(DEVICE)
print('✅ Discriminator ready!')
print(f'   Parameters: {sum(p.numel() for p in D.parameters()):,}')
print(D)

In [ ]:
# CELL 6 — Loss & Optimizers
criterion   = nn.BCELoss()    # Binary Cross Entropy for real/fake

# ✅ Separate optimizers for G and D
opt_G = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

# Fixed noise to track Generator progress over epochs
fixed_noise = torch.randn(16, NOISE_DIM).to(DEVICE)

print('✅ Loss function : BCELoss')
print('✅ Optimizer G   : Adam (lr=0.0002)')
print('✅ Optimizer D   : Adam (lr=0.0002)')
print('✅ Fixed noise   : ready for image saving')

In [ ]:
# CELL 7 — Helper: Save Generated Images
def save_generated_images(epoch, generator, noise):
    generator.eval()
    with torch.no_grad():
        fake_imgs = generator(noise).cpu()

    fig, axes = plt.subplots(2, 8, figsize=(14, 4))
    fig.suptitle(f'Generated Images — Epoch {epoch}',
                 fontsize=12, fontweight='bold')
    for i, ax in enumerate(axes.flat):
        img = fake_imgs[i].squeeze().numpy()
        img = (img + 1) / 2          # scale from [-1,1] to [0,1]
        ax.imshow(img, cmap='gray')
        ax.axis('off')
    plt.tight_layout()
    fname = f'./generated_images/epoch_{epoch:03d}.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.close()
    generator.train()
    return fname

print('✅ Image saving function ready!')

In [ ]:
# CELL 8 — GAN Training Loop
# Alternating: Train D first, then Train G

d_losses, g_losses = [], []
saved_files        = []
start              = time.time()

print(f'🚀 Training GAN for {EPOCHS} epochs...')
print(f'   Saving images every {SAVE_EVERY} epochs')
print('─' * 55)
print(f'{"Epoch":>5} | {"D Loss":>8} | {"G Loss":>8} | {"D(real)":>7} | {"D(fake)":>7}')
print('─' * 55)

for epoch in range(1, EPOCHS + 1):
    epoch_d_loss = 0
    epoch_g_loss = 0
    d_real_acc   = 0
    d_fake_acc   = 0
    n_batches    = 0

    for real_imgs, _ in loader:
        real_imgs = real_imgs.to(DEVICE)
        batch     = real_imgs.size(0)

        # ── Labels ──────────────────────────────────────────
        real_labels = torch.ones(batch, 1).to(DEVICE)   # 1 = real
        fake_labels = torch.zeros(batch, 1).to(DEVICE)  # 0 = fake

        # ══════════════════════════════════════════
        # STEP 1 — Train Discriminator
        # Goal: correctly label real=1 and fake=0
        # ══════════════════════════════════════════
        D.zero_grad()

        # Real images → D should say 1
        out_real  = D(real_imgs)
        loss_real = criterion(out_real, real_labels)

        # Fake images → D should say 0
        noise     = torch.randn(batch, NOISE_DIM).to(DEVICE)
        fake_imgs = G(noise).detach()    # detach so G is not updated
        out_fake  = D(fake_imgs)
        loss_fake = criterion(out_fake, fake_labels)

        loss_D = loss_real + loss_fake   # total D loss
        loss_D.backward()
        opt_D.step()

        # ══════════════════════════════════════════
        # STEP 2 — Train Generator
        # Goal: fool D into thinking fake = real
        # ══════════════════════════════════════════
        G.zero_grad()

        noise     = torch.randn(batch, NOISE_DIM).to(DEVICE)
        fake_imgs = G(noise)
        out_fake  = D(fake_imgs)

        # G wants D to say 1 (real) for its fake images
        loss_G = criterion(out_fake, real_labels)
        loss_G.backward()
        opt_G.step()

        # Track metrics
        epoch_d_loss += loss_D.item()
        epoch_g_loss += loss_G.item()
        d_real_acc   += out_real.mean().item()
        d_fake_acc   += out_fake.mean().item()
        n_batches    += 1

    # Average losses per epoch
    avg_d = epoch_d_loss / n_batches
    avg_g = epoch_g_loss / n_batches
    avg_dr = d_real_acc  / n_batches
    avg_df = d_fake_acc  / n_batches

    d_losses.append(avg_d)
    g_losses.append(avg_g)

    # Save images every N epochs
    if epoch % SAVE_EVERY == 0 or epoch == 1:
        fname = save_generated_images(epoch, G, fixed_noise)
        saved_files.append(fname)
        print(f'{epoch:>5} | {avg_d:>8.4f} | {avg_g:>8.4f} | {avg_dr:>7.3f} | {avg_df:>7.3f}  ✅ saved')
    else:
        print(f'{epoch:>5} | {avg_d:>8.4f} | {avg_g:>8.4f} | {avg_dr:>7.3f} | {avg_df:>7.3f}')

elapsed = time.time() - start
print('─' * 55)
print(f'✅ Training complete in {elapsed/60:.1f} minutes!')

In [ ]:
# CELL 9 — Plot GAN Loss Curves
epochs_list = range(1, EPOCHS + 1)

plt.figure(figsize=(10, 4))
plt.plot(epochs_list, d_losses, label='Discriminator Loss',
         color='tomato',    linewidth=2)
plt.plot(epochs_list, g_losses, label='Generator Loss',
         color='steelblue', linewidth=2)
plt.axhline(y=0.693, color='gray', linestyle='--',
            alpha=0.7, label='Ideal Loss (0.693)')
plt.title('GAN Training Loss Curves', fontsize=13, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(fontsize=11); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('./outputs/gan_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: gan_loss_curves.png')

In [ ]:
# CELL 10 — Show Image Quality Progression
# Compare generated images: Epoch 1 vs middle vs final
print('📊 Image Quality Progression:')

fig, axes = plt.subplots(len(saved_files), 8, figsize=(14, 2.5 * len(saved_files)))
if len(saved_files) == 1:
    axes = [axes]

for row, fpath in enumerate(saved_files):
    img_grid = plt.imread(fpath)
    axes[row][0].imshow(img_grid)
    axes[row][0].set_title(os.path.basename(fpath), fontsize=8)
    for ax in axes[row]:
        ax.axis('off')

plt.suptitle('GAN Output Progression (Epoch 1 → Final)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('./outputs/progression.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: progression.png')

In [ ]:
# CELL 11 — Show Final Generated Images
G.eval()
with torch.no_grad():
    sample_noise = torch.randn(16, NOISE_DIM).to(DEVICE)
    final_imgs   = G(sample_noise).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle(f'Final Generated Images (Epoch {EPOCHS})',
             fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    img = final_imgs[i].squeeze().numpy()
    img = (img + 1) / 2
    ax.imshow(img, cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.savefig('./outputs/final_generated.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: final_generated.png')

In [ ]:
# CELL 12 — Failure Mode Discussion + Final Summary
print('='*55)
print('         GAN TRAINING SUMMARY')
print('='*55)
print(f'  Epochs trained       : {EPOCHS}')
print(f'  Final D Loss         : {d_losses[-1]:.4f}')
print(f'  Final G Loss         : {g_losses[-1]:.4f}')
print(f'  Training time        : {elapsed/60:.1f} minutes')
print(f'  Images saved         : {len(saved_files)} checkpoints')
print('='*55)

print('''
📌 FAILURE MODE OBSERVED: Mode Collapse
─────────────────────────────────────────
What happened:
  Generator started producing very similar
  looking images (same texture/pattern)
  instead of diverse Fashion-MNIST items.

Why it happens:
  Generator found one image that always
  fools the Discriminator — so it kept
  repeating it instead of exploring more.

Mitigation attempted:
  1. Added Dropout(0.3) in Discriminator
     → stops D from becoming too strong
  2. Used LeakyReLU instead of ReLU
     → allows small gradients to flow
  3. Separate Adam optimizers for G and D
     → balanced learning between both
  4. Label smoothing (real=1.0, fake=0.0)
     → prevents D from being overconfident
''')

print('📁 ALL OUTPUT FILES:')
print('─'*40)
for f in os.listdir('./outputs'):
    print(f'  ✅ outputs/{f}')
print()
print(f'  📸 generated_images/ → {len(os.listdir("./generated_images"))} image files')
print('─'*40)
print('\n🎉 Part C Complete! All 3 parts done!')
print('   → Part A: CNN    ✅')
print('   → Part B: RNN    ✅')
print('   → Part C: GAN    ✅')